In [16]:
%pip install networkx pandas

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import networkx as nx

# 1. CONFIGURATION
ESSENTIAL_FILE = 'Data - List of essential proteins of saccharomyces cerevisiae.csv'
PPI_FILE = '4932.protein.links.v12.0.txt'

EXCLUDE_ESSENTIALS = True   # toggle: True -> drop any edge touching an essential protein
THRESHOLD = 750              # STRING combined_score cutoff (0-1000 scale); 750 = "high confidence"

# 2. LOADING / CLEANING SUBROUTINES

def load_essential_set(essential_path):
    essential_df = pd.read_csv(essential_path)
    # Assumes the protein identifier is in the 2nd column, matching the
    # original script's `essential_df.iloc[:, 1]`.
    return set(essential_df.iloc[:, 1].astype(str).str.strip())

def strip_species_prefix(protein_id, prefix='4932.'):
    return protein_id.replace(prefix, '')


def load_raw_ppi_graph(ppi_path):
    with open(ppi_path, 'r') as f:
        next(f)  # skip header line
        G_raw = nx.read_weighted_edgelist(f, nodetype=str)
    return G_raw


def clean_node_names(G_raw, prefix='4932.'):
    mapping = {node: strip_species_prefix(node, prefix) for node in G_raw.nodes()}
    return nx.relabel_nodes(G_raw, mapping)


# 3. NETWORK CONSTRUCTION SUBROUTINE

def build_yeast_network(ppi_path, essential_path,
                         exclude_essential_toggle=False,
                         threshold_cutoff=750,
                         export_path=None):
    
    essential_set = load_essential_set(essential_path)

    G_raw = load_raw_ppi_graph(ppi_path)
    G_clean = clean_node_names(G_raw)

    if export_path:
        nx.to_pandas_edgelist(G_clean).to_csv(export_path, index=False)

    # Build the filtered, unweighted graph
    G0 = nx.Graph()
    for u, v, data in G_clean.edges(data=True):
        if data['weight'] < threshold_cutoff:
            continue
        if exclude_essential_toggle and (u in essential_set or v in essential_set):
            continue
        G0.add_edge(u, v)   # no weight attribute added -> unweighted graph

    # Reduce to the largest connected component
    largest_cc, total_nodes, nodes_in_largest, nodes_discarded = largest_connected_component(G0)

    print(f"--- NETWORK METRICS (Exclude Essentials = {exclude_essential_toggle}) ---")
    print(f"Filtered network: {total_nodes} proteins, {G0.number_of_edges()} links.")
    print(f"Largest Connected Component size: {nodes_in_largest} proteins.")
    print(f"Nodes discarded to ensure a connected graph: {nodes_discarded}\n")

    G_final = G0.subgraph(largest_cc).copy()
    return G_final, essential_set


# ---------------------------------------------------------------------------
# 4. REQUIRED SUBROUTINES
# ---------------------------------------------------------------------------

def largest_connected_component(G):
    if G.number_of_nodes() == 0:
        return set(), 0, 0, 0

    components = list(nx.connected_components(G))
    largest_cc = max(components, key=len)

    total_nodes = G.number_of_nodes()
    nodes_in_largest = len(largest_cc)
    nodes_discarded = total_nodes - nodes_in_largest

    return largest_cc, total_nodes, nodes_in_largest, nodes_discarded


def shortest_path_length(G, protein_a, protein_b):
    if protein_a not in G:
        print(f"Warning: '{protein_a}' not found in the network.")
        return None
    if protein_b not in G:
        print(f"Warning: '{protein_b}' not found in the network.")
        return None
    try:
        return nx.shortest_path_length(G, source=protein_a, target=protein_b)
    except nx.NetworkXNoPath:
        print(f"Warning: no path exists between '{protein_a}' and '{protein_b}'.")
        return None

def is_essential(protein_id, essential_set):
    return protein_id in essential_set

# 5. RUN THE PIPELINE
if __name__ == '__main__':
    G, essential_set = build_yeast_network(
        PPI_FILE, ESSENTIAL_FILE,
        exclude_essential_toggle=EXCLUDE_ESSENTIALS,
        threshold_cutoff=THRESHOLD,
        export_path='ppi_clean_no_prefix.csv'
    )

    # --- Example usage of the subroutines ---

    # 1. Shortest path length between two proteins (replace with real IDs
    #    from your cleaned network, e.g. two YAL/YBL-style systematic names
    #    or STRING protein IDs with the 4932. prefix already stripped).
    protein_a, protein_b = 'YMR190C', 'YLR234W'
    d = shortest_path_length(G, protein_a, protein_b)
    print(f"Shortest path length between {protein_a} and {protein_b}: {d}")

    # 2. Largest connected component of an arbitrary subgraph (here, the
    #    full G itself, which by construction IS already its own LCC).
    lcc, total, in_lcc, discarded = largest_connected_component(G)
    print(f"LCC check: {in_lcc} of {total} nodes retained, {discarded} discarded.")

    # 3. Essentiality check for a candidate knockout protein.
    candidate = 'YMR190C'
    if is_essential(candidate, essential_set):
        print(f"{candidate} is ESSENTIAL -- do not suggest as a knockout target.")
    else:
        print(f"{candidate} is non-essential -- safe to suggest as a knockout target.")

--- NETWORK METRICS (Exclude Essentials = True) ---
Filtered network: 4377 proteins, 41159 links.
Largest Connected Component size: 4209 proteins.
Nodes discarded to ensure a connected graph: 168

Shortest path length between YMR190C and YLR234W: 1
LCC check: 4209 of 4209 nodes retained, 0 discarded.
YMR190C is non-essential -- safe to suggest as a knockout target.


In [ ]:
genes_of_interest = ['YMR190C', 'YLR234W', 'YPL024W', 'YIR002C', 'YOR033C', 'YAR007C']

# Include first-degree neighbors
neighborhood = set(genes_of_interest)
for gene in genes_of_interest:
    if gene in G:
        neighborhood.update(G.neighbors(gene))

subG = G.subgraph(neighborhood)
lcc, total, in_lcc, discarded = largest_connected_component(subG)
print(f"Neighborhood check: {in_lcc} of {total} nodes in the largest component, {discarded} discarded.")

Neighborhood check: 107 of 107 nodes in the largest component, 0 discarded.


In [19]:
import networkx as nx

def compute_centralities(G, katz_alpha=None, eigenvector_max_iter=1000):
    centralities = {}

# Degree centrality: fraction of all other nodes directly connected to
    centralities['degree'] = nx.degree_centrality(G)

# Eigenvector centrality: important if connected to other important nodes
    try:
        centralities['eigenvector'] = nx.eigenvector_centrality(
            G, max_iter=eigenvector_max_iter
        )
    except nx.PowerIterationFailedConvergence:
        print("Warning: eigenvector centrality did not converge.")
        centralities['eigenvector'] = {}

# Katz centrality: like eigenvector, but with a baseline score for every node
    if katz_alpha is None:
        lambda_max = max(nx.adjacency_spectrum(G)).real
        katz_alpha = 0.9 / lambda_max if lambda_max > 0 else 0.1
    centralities['katz'] = nx.katz_centrality(
        G, alpha=katz_alpha, max_iter=eigenvector_max_iter
    )

# PageRank: random-walk-based, similar spirit to eigenvector centrality
    centralities['pagerank'] = nx.pagerank(G)

# Betweenness: fraction of shortest paths (between other node pairs) passing through this node
    centralities['betweenness'] = nx.betweenness_centrality(G)

# Subgraph centrality: based on closed walks starting/ending at the node
    centralities['subgraph'] = nx.subgraph_centrality(G)

# Closeness: inverse of average shortest-path distance to all other nodes
    centralities['closeness'] = nx.closeness_centrality(G)

    return centralities

def top_n_nodes(centrality_scores, n=5):
    return sorted(centrality_scores.items(), key=lambda item: item[1], reverse=True)[:n]

centralities = compute_centralities(G)

for measure_name, scores in centralities.items():
    print(f"\nTop 5 by {measure_name}:")
    for node, score in top_n_nodes(scores, n=5):
        print(f"  {node}: {score:.4f}")


Top 5 by degree:
  YGR220C: 0.0570
  YLR344W: 0.0554
  YNL284C: 0.0547
  YGR214W: 0.0544
  YGR118W: 0.0542

Top 5 by eigenvector:
  YIL018W: 0.0873
  YLR344W: 0.0873
  YFR031C-A: 0.0872
  YGR118W: 0.0872
  YPR132W: 0.0872

Top 5 by katz:
  YLR344W: 0.0785
  YGR118W: 0.0783
  YIL018W: 0.0783
  YPR132W: 0.0783
  YFR031C-A: 0.0783

Top 5 by pagerank:
  YBR010W: 0.0018
  YLL039C: 0.0014
  YNL030W: 0.0012
  YER095W: 0.0012
  YDR477W: 0.0011

Top 5 by betweenness:
  YLL039C: 0.0451
  YBR010W: 0.0426
  YLL013C: 0.0295
  YJR066W: 0.0261
  Q0120: 0.0213

Top 5 by subgraph:
  YIL018W: 5809440166718606857883345871982448403549612557681449101834357571584.0000
  YLR344W: 5807811082187056078204353431716003720527832596487221313769589178368.0000
  YFR031C-A: 5799949948869618929861279872076003431098600322932349534259707379712.0000
  YGR118W: 5798485182002180398441800281944128550117584760806180004618777395200.0000
  YPR132W: 5798025718511213533847657865502861281910047880704750613511020740608.0000

Top 5

In [20]:
# The centrality measures: degree/eigenvector/Katz all have many ribosomal proteins in their 
# top 5, this is because every ribosomal protein has many high-confidence STRING interactions 
# with every other ribosomal protein. So maybe we should consider using betweenness centrality 
# since it looks for bridge proteins between different functional modules, which is less likely 
# to just return the ribosome every time. -- this result is fairly invariant to excluding essential
# proteins or not.

# I also think closeness isn't the best idea since given we have a large, sparse PPI network dominated 
# by a few dense hub clusters (like the ribosome), the closeness centrality will reflect a proteins
# general position in the overall network topology rather than anything specific to its function


# Top 5 by degree:
 # YNL178W: 0.0746
 # YDR064W: 0.0725
 # YJR123W: 0.0707
 # YOR063W: 0.0681
 # YGL123W: 0.0678

Top 5 by eigenvector:
 # YDR064W: 0.0806
 # YJR123W: 0.0794
 # YCR031C: 0.0783
 # YNL178W: 0.0780
 # YDR025W: 0.0769

Top 5 by katz:
 # YDR064W: 0.0732
 # YJR123W: 0.0721
 # YNL178W: 0.0715
 # YCR031C: 0.0709
 # YGL123W: 0.0697

Top 5 by pagerank:
  #YFL039C: 0.0014
 # YBR010W: 0.0013
 # YFL039C: 0.3419
 # YOR063W: 0.3392
  # YLR167W: 0.3377
  # YOL127W: 0.3377

SyntaxError: invalid syntax (2833006532.py, line 20)

In [21]:
import itertools
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import numpy as np

def plot_centrality_pairs(centralities, node_order=None):
    measures = list(centralities.keys())
    if node_order is None:
        node_order = list(centralities[measures[0]].keys())

    pairs = list(itertools.combinations(measures, 2))
    n_pairs = len(pairs)
    n_cols = 4
    n_rows = -(-n_pairs // n_cols)  # ceiling division

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    for ax, (m1, m2) in zip(axes, pairs):
        x = [centralities[m1][node] for node in node_order]
        y = [centralities[m2][node] for node in node_order]
        ax.scatter(x, y, s=8, alpha=0.4)
        ax.set_xlabel(m1)
        ax.set_ylabel(m2)

    # hide unused subplots if pairs don't fill the grid exactly
    for ax in axes[n_pairs:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()


def centrality_correlation_matrix(centralities, node_order=None):
    import pandas as pd

    measures = list(centralities.keys())
    if node_order is None:
        node_order = list(centralities[measures[0]].keys())

    corr_matrix = pd.DataFrame(index=measures, columns=measures, dtype=float)

    for m1, m2 in itertools.combinations_with_replacement(measures, 2):
        x = [centralities[m1][node] for node in node_order]
        y = [centralities[m2][node] for node in node_order]
        rho, _ = spearmanr(x, y)
        corr_matrix.loc[m1, m2] = rho
        corr_matrix.loc[m2, m1] = rho

    return corr_matrix